In [811]:
board = [
[0,0,0],
[0,0,0],
[0,0,0]
]



In [812]:
board

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [813]:
board[1][1] = 1

In [814]:
board

[[0, 0, 0], [0, 1, 0], [0, 0, 0]]

In [815]:
empty = 0
X = 1 
O = -1

In [816]:
board[0][0] =X

In [817]:
board[1][1] = O

In [818]:
board

[[1, 0, 0], [0, -1, 0], [0, 0, 0]]

In [819]:
def invalid(a,b):
    if board[a][b] == X or board[a][b] == O:
        return True

    return False

In [820]:
def move(current_player,a,b):
  if current_player == X:
    board[a][b] = 1
  else:
     board[a][b] = -1


In [821]:
def clear(board):
  for i in range (3):
    for j in range (3):
      board[i][j] = 0

In [822]:
clear(board)

In [823]:
board  

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [824]:
current_player = X
move(X,0,0)

In [825]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [826]:
board

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]

In [827]:
def check_win(board):
  if board[0][0] == board[0][1] == board [0][2] == 1 or board[1][0] == board[1][1] == board [1][2] == 1 or board[2][0] == board[2][1] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][0] == board [2][0] == 1 or board[0][1] == board[1][1] == board [2][1] == 1 or board[0][2] == board[1][2] == board [2][2] == 1:
    return X
  elif board[0][0] == board[1][1] == board[2][2] == 1:
    return X
  elif board[0][2] == board[1][1] == board[2][0] == 1:
    return X
  elif board[0][0] == board[0][1] == board [0][2] == -1 or board[1][0] == board[1][1] == board [1][2] == -1 or board[2][0] == board[2][1] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][0] == board [2][0] == -1 or board[0][1] == board[1][1] == board [2][1] == -1 or board[0][2] == board[1][2] == board [2][2] == -1:
    return O
  elif board[0][0] == board[1][1] == board[2][2] == -1:
    return O
  elif board[0][2] == board[1][1] == board[2][0] == -1:
    return O
  else:
    return None

In [828]:
Draw = "Draw"
def check_draw(board):
  board_full = True
  for i in range (3):
    for j in range (3):
      if board[i][j] == 0:
        board_full= False
 
  if check_win(board) == None and board_full:
    return Draw
  

In [829]:
import random
def random_move():

  row = random.randint(0,2)
  column = random.randint(0,2)
  return row,column

In [830]:
clear(board)
print(board)


[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [831]:
def flatten(board):
  new_board = []
  for i in range (3):
    for j in range (3):
      new_board.append(board[i][j])
  return new_board 


In [832]:
def indexing(row,column):
  index = row*3 + column
  return index


In [833]:
X_train = []
y_train = []


In [834]:
for game in range (5000):
  clear(board)
  history = []
  current_player = X
  while (check_win(board)!= X and check_win(board)!= O) and check_draw(board)!=Draw:
    board_snapshot = [row[:] for row in board]
    row , column = random_move()

    while invalid(row,column):
      row , column = random_move()
    if current_player == X:
      history.append([board_snapshot,
                (row,column),
                X])
      move(X, row, column)
      
      current_player = O

    else:
      history.append([board_snapshot,
                (row,column),
                O])
      move(O,row,column)
      
      current_player = X

  if check_win(board) == X:
    reward = 1

  elif check_win(board) == O:
    reward = -1

  elif check_draw(board) == Draw:
    reward = 0
  for i in history:
    i.append(reward)
  winner_moves = []
  for i in range (len(history)):
    if (reward==1 and history[i][2] == 1):
      winner_moves.append(history[i])
    elif (reward==-1 and history[i][2] == -1):
       winner_moves.append(history[i])

  if len(winner_moves) > 0:
    last_move = winner_moves[-3:]

    for moves in last_move:
      X_train.append(flatten(moves[0]))
      y_train.append(indexing(*moves[1]))  





In [835]:
#X_train

In [836]:
#y_train

In [837]:
print(len(X_train))
print(len(y_train))

13104
13104


In [838]:
import torch

In [839]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)

In [840]:
print(X_train.shape)
print(y_train.shape)

torch.Size([13104, 9])
torch.Size([13104])


# Defining the model

In [841]:
# define NN class
import torch.nn as nn
class MyNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()
    self.model = nn.Sequential(
      nn.Linear(num_features, 32),
      nn.ReLU(),
      nn.Linear(32,32),
      nn.ReLU(),
      nn.Linear(32, 9)
    )

  def forward(self, x):
      return self.model(x)

  

In [842]:
model = MyNN(9)

In [843]:
criterion = nn.CrossEntropyLoss()

In [844]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [845]:
X_train = X_train.float()
epochs = 100

# training loop

In [846]:
for epoch in range(epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 2.1963460445404053
Epoch: 2, Loss: 2.1948695182800293
Epoch: 3, Loss: 2.193441867828369
Epoch: 4, Loss: 2.1920533180236816
Epoch: 5, Loss: 2.1906979084014893
Epoch: 6, Loss: 2.1893792152404785
Epoch: 7, Loss: 2.188096284866333
Epoch: 8, Loss: 2.1868488788604736
Epoch: 9, Loss: 2.1856400966644287
Epoch: 10, Loss: 2.184459924697876
Epoch: 11, Loss: 2.1833081245422363
Epoch: 12, Loss: 2.1821823120117188
Epoch: 13, Loss: 2.1810836791992188
Epoch: 14, Loss: 2.180009365081787
Epoch: 15, Loss: 2.1789588928222656
Epoch: 16, Loss: 2.1779282093048096
Epoch: 17, Loss: 2.1769189834594727
Epoch: 18, Loss: 2.1759262084960938
Epoch: 19, Loss: 2.1749494075775146
Epoch: 20, Loss: 2.1739869117736816
Epoch: 21, Loss: 2.1730401515960693
Epoch: 22, Loss: 2.1721086502075195
Epoch: 23, Loss: 2.171191930770874
Epoch: 24, Loss: 2.1702911853790283
Epoch: 25, Loss: 2.169405698776245
Epoch: 26, Loss: 2.1685304641723633
Epoch: 27, Loss: 2.167667865753174
Epoch: 28, Loss: 2.1668126583099365
Epoch: 2

In [847]:
sample = X_train[0]

In [848]:
sample = sample.unsqueeze(0)

In [849]:
output = model(sample)
output

tensor([[ 0.5128, -0.0224, -0.1952, -0.2474,  0.2444, -0.4488,  0.2170, -0.0067,
          0.3351]], grad_fn=<AddmmBackward0>)

In [850]:
prediction = torch.argmax(output)
print(prediction)

tensor(0)


In [851]:
clear(board)
print(board)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [852]:

ai_wins = 0
random_wins = 0
draws = 0

for game in range(100):
  clear(board)
  current_player = X
  while (check_win(board)!=X and check_win(board)!=O) and check_draw(board)!=Draw:
    if current_player == X:
      input = torch.tensor(flatten(board))
      input = input.float()
      input = input.unsqueeze(0)
      output = model(input)

      ranked_moves = torch.argsort(output, descending=True)

      for index in ranked_moves[0]:
        index = index.item()
        row = index // 3
        column = index % 3
        if not invalid(row,column):
         move(X,row,column)
        current_player = O
        break
    
    else:

      row,column = random_move()

      while invalid(row,column):
        row,column = random_move()
      
      move(O,row,column)

      current_player = X
    
    
  if check_win(board) == X:
    ai_wins += 1

  elif check_win(board) == O:
    random_wins += 1

  else:
    draws += 1

In [853]:

print(ai_wins)
print(random_wins)
print(draws)

54
46
0
